# Stock Market Analysis with Python

This notebook shows the **analysis behind the MarketLens web app**, step by step, using real historical prices for **Apple (AAPL)** and **NVIDIA (NVDA)**.

**What you will do**
1. Import the Python libraries
2. Retrieve historical stock data
3. Clean and process it with **Pandas**
4. Calculate returns and statistics with **NumPy**
5. Calculate moving averages (and RSI)
6. Measure historical volatility
7. Draw charts with **Matplotlib**
8. Compare Apple and NVIDIA

**How to run it:** install the requirements (`pip install -r requirements.txt`), start Jupyter from the project folder (`jupyter lab`), open this file, and choose *Run > Run All Cells*.
The notebook uses the same data provider settings as the app (see `.env.example`).

> Educational project. Nothing here is investment advice, and past performance does not guarantee future results.

## 1. Importing libraries

- **pandas** keeps data in tables called *DataFrames*.
- **numpy** does fast maths on arrays of numbers.
- **matplotlib** draws charts.
- The `modules` folder contains the project's own code, so the notebook and the web app use exactly the same functions.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # lets the notebook import the project's `modules` package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modules.data_fetcher import get_service
from modules import data_processor as dp
from modules import technical_indicators as ti

pd.options.display.float_format = "{:,.4f}".format
plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.3})

## 2. Retrieving historical stock data

`get_service().history(symbol, period)` asks the configured data provider (Yahoo Finance by default) for **daily OHLCV** data:
**O**pen, **H**igh, **L**ow, **C**lose prices and trading **V**olume.

The function below also saves a CSV copy in the `data/` folder. If a later download fails (for example, no internet), it reuses that saved copy of **real data** from the earlier download. It never invents prices.

In [ ]:
DATA_DIR = Path.cwd().parent / "data"
SYMBOLS = ["AAPL", "NVDA"]
PERIOD = "2y"          # try "1y", "5y", ...

def load_prices(symbol, period=PERIOD):
    """Download daily prices, keep a CSV copy, and fall back to that copy if the download fails."""
    result = get_service().history(symbol, period)
    csv_path = DATA_DIR / f"{symbol}_history.csv"
    if result.ok:
        DATA_DIR.mkdir(exist_ok=True)
        result.data.to_csv(csv_path)
        return result.data, "download" + (" (older cached copy: refresh failed)" if result.stale else "")
    if csv_path.exists():
        saved = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        return saved, f"saved CSV from an earlier download (new download failed: {result.message})"
    raise RuntimeError(f"Could not download {symbol} and no saved copy exists: {result.message}")

prices = {}
for symbol in SYMBOLS:
    prices[symbol], source = load_prices(symbol)
    df = prices[symbol]
    print(f"{symbol}: {len(df)} rows from {df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}  [{source}]")

prices["AAPL"].tail()

## 3. Cleaning and processing data with Pandas

Real data is rarely perfect. Before analysing it we check for **missing values**, **duplicate dates** and **wrong data types**.
`dp.clean_ohlcv()` (in `modules/data_processor.py`) fixes these problems:

- convert every column to numbers,
- sort by date and remove duplicate dates,
- drop rows without a valid closing price,
- fill missing Open/High/Low with the Close, and missing Volume with 0.

In [ ]:
aapl_raw = prices["AAPL"]
print("Data types:")
print(aapl_raw.dtypes, "\n")
print("Missing values per column:")
print(aapl_raw.isna().sum(), "\n")
print("Duplicate dates:", aapl_raw.index.duplicated().sum())

clean = {symbol: dp.clean_ohlcv(df) for symbol, df in prices.items()}
clean["AAPL"].describe()

## 4. Numerical calculations with NumPy: daily returns

A **daily return** tells us how much the price changed from one day to the next:

$$ r_t = \frac{P_t}{P_{t-1}} - 1 $$

Below we calculate it twice: once with Pandas (`pct_change`) and once with plain NumPy array maths, and check that both agree.

In [ ]:
close = clean["AAPL"]["Close"]

returns_pandas = close.pct_change().dropna()              # the Pandas way

values = close.to_numpy()                                  # the NumPy way
returns_numpy = values[1:] / values[:-1] - 1

print("Both methods agree:", np.allclose(returns_pandas.to_numpy(), returns_numpy))
print(f"Average daily return : {returns_numpy.mean():.4%}")
print(f"Best day / worst day : {returns_numpy.max():.2%} / {returns_numpy.min():.2%}")
print(f"Highest / lowest price in the period: ${clean['AAPL']['High'].max():,.2f} / ${clean['AAPL']['Low'].min():,.2f}")

## 5. Moving averages (and RSI)

A **moving average** smooths the price so the trend is easier to see.

- **SMA** (simple): the plain average of the last *N* closes.
- **EMA** (exponential): gives recent prices more weight.

The chart also shows the **RSI**, a 0 to 100 momentum score (above 70 is often called overbought, below 30 oversold; it is a rule of thumb, not a prediction).

In [ ]:
aapl = clean["AAPL"].copy()
aapl["SMA 20"] = aapl["Close"].rolling(window=20).mean()   # Pandas rolling window
aapl["SMA 50"] = ti.sma(aapl["Close"], 50)                  # same idea, using the project's helper
aapl["EMA 20"] = ti.ema(aapl["Close"], 20)
aapl["RSI 14"] = ti.rsi(aapl["Close"], 14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(aapl.index, aapl["Close"], label="Close", linewidth=1.5)
ax1.plot(aapl.index, aapl["SMA 20"], label="SMA 20")
ax1.plot(aapl.index, aapl["SMA 50"], label="SMA 50")
ax1.plot(aapl.index, aapl["EMA 20"], label="EMA 20", linestyle="--")
ax1.set_title("Apple (AAPL): price and moving averages")
ax1.set_ylabel("Price (USD)")
ax1.legend()
ax2.plot(aapl.index, aapl["RSI 14"], color="purple")
ax2.axhline(70, color="red", linestyle="--", linewidth=1)
ax2.axhline(30, color="green", linestyle="--", linewidth=1)
ax2.set_ylabel("RSI (14)")
ax2.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 6. Historical volatility

**Volatility** measures how much returns vary. It is the *standard deviation* of daily returns.
To compare it with yearly figures we **annualize** it by multiplying by the square root of 252 (the number of trading days in a year):

$$ \sigma_{annual} = \sigma_{daily} \times \sqrt{252} $$

The chart shows a **rolling 30-day volatility**, so we can see how calm or turbulent each period was.

In [ ]:
rolling_vol = pd.DataFrame({
    symbol: clean[symbol]["Close"].pct_change().rolling(30).std() * np.sqrt(252) * 100
    for symbol in SYMBOLS
})

rolling_vol.plot(title="Rolling 30-day annualized volatility (%)", ylabel="Volatility (%)")
plt.show()

for symbol in SYMBOLS:
    stats = dp.summary_stats(clean[symbol])
    print(f"{symbol}: annualized volatility over the whole period = {stats['annual_volatility']:.1%}")

## 7. Visualizing results with Matplotlib

Two classic views for each stock: the **closing price with trading volume**, and a **histogram of daily returns** (how often each size of daily move happened).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex="col")
for col, symbol in enumerate(SYMBOLS):
    df = clean[symbol]
    axes[0, col].plot(df.index, df["Close"], linewidth=1.5)
    axes[0, col].set_title(f"{symbol} closing price (USD)")
    axes[1, col].bar(df.index, df["Volume"] / 1e6, width=1.0, color="steelblue")
    axes[1, col].set_ylabel("Volume (millions)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, symbol in zip(axes, SYMBOLS):
    r = clean[symbol]["Close"].pct_change().dropna() * 100
    ax.hist(r, bins=40, edgecolor="white")
    ax.axvline(r.mean(), color="orange", linewidth=2, label=f"average {r.mean():.2f}%")
    ax.set_title(f"{symbol}: distribution of daily returns")
    ax.set_xlabel("Daily return (%)")
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Comparing Apple and NVIDIA

Prices are in different ranges, so we **normalize** them: each series is rescaled to start at 100. Then a value of 150 means "up 50% since the start".
We also compare return, risk (volatility), worst fall (maximum drawdown) and how closely the two stocks move together (correlation).

In [ ]:
common = pd.concat({symbol: clean[symbol]["Close"] for symbol in SYMBOLS}, axis=1).dropna()
normalized = common / common.iloc[0] * 100

normalized.plot(title="Apple vs NVIDIA: performance since the start of the period (start = 100)", ylabel="Index (start = 100)")
plt.axhline(100, color="gray", linestyle=":")
plt.show()

rows = {}
for symbol in SYMBOLS:
    s = dp.summary_stats(clean[symbol])
    rows[symbol] = {
        "Total return": s["total_return"], "Average daily return": s["avg_daily_return"],
        "Annualized volatility": s["annual_volatility"], "Maximum drawdown": s["max_drawdown"],
        "Best day": s["best_day"], "Worst day": s["worst_day"],
    }
comparison = pd.DataFrame(rows).T
display(comparison.map("{:.2%}".format))

correlation = common.pct_change().dropna().corr().iloc[0, 1]
print(f"Correlation of daily returns (AAPL vs NVDA): {correlation:.2f}   (1 = move together, 0 = unrelated, -1 = opposite)")

## 9. Conclusions and limitations

- Pandas and NumPy turn raw price tables into **returns, averages and risk measures** with only a few lines of code.
- Matplotlib makes patterns (trends, volatility, differences between stocks) visible.
- **Limitations:** the results describe only the chosen period; free data may be delayed or contain errors; indicators describe the past and do not predict the future.

**Sources:** [Yahoo Finance](https://finance.yahoo.com/) · [pandas docs](https://pandas.pydata.org/docs/) · [NumPy docs](https://numpy.org/doc/) · [Matplotlib docs](https://matplotlib.org/stable/) · [SEC Investor.gov](https://www.investor.gov/)